In [ ]:
import os
import zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16  # Import VGG16
from tensorflow.keras import layers, models
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO

# Define paths
zip_file_path = "Final_ODIR_DataSet.zip"  # Path to ZIP file

# Open the ZIP file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    # List of file paths inside the ZIP
    file_list = zip_ref.namelist()

# Paths to directories inside the ZIP
train_dir = [f for f in file_list if f.startswith("Final_ODIR_DataSet/training/")]
val_dir = [f for f in file_list if f.startswith("Final_ODIR_DataSet/validate/")]
test_dir = [f for f in file_list if f.startswith("Final_ODIR_DataSet/test/")]

# Helper function to load images from the ZIP file
def load_images_from_zip(zip_path, file_paths, img_size):
    images = []
    labels = []
    label_map = {}
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for file_path in file_paths:
            if file_path.endswith('.jpg') or file_path.endswith('.png'):
                # Extract label from folder structure
                label = file_path.split('/')[-2]
                if label not in label_map:
                    label_map[label] = len(label_map)

                # Load and preprocess image
                img_data = zip_ref.read(file_path)
                img = Image.open(BytesIO(img_data)).resize(img_size)
                images.append(np.array(img) / 255.0)
                labels.append(label_map[label])
    return np.array(images), tf.keras.utils.to_categorical(labels), label_map

# Image Parameters for VGG16
img_size = (224, 224)  # VGG16 requires 224x224 images

# Load datasets
x_train, y_train, label_map = load_images_from_zip(zip_file_path, train_dir, img_size)
x_val, y_val, _ = load_images_from_zip(zip_file_path, val_dir, img_size)
x_test, y_test, _ = load_images_from_zip(zip_file_path, test_dir, img_size)

# Load Pretrained VGG16 Model
vgg_base = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze Pretrained Layers
vgg_base.trainable = False

# Build the Model
model = models.Sequential([
    vgg_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(len(label_map), activation="softmax")
])

# Compile the Model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the Model for 100 epochs
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=100,  # Changed to 100 epochs
    batch_size=32
)

# Evaluate on Test Data
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Save the Model
model.save("Final_ODIR_Model_VGG16_100epochs.h5")

# Confusion Matrix and Classification Report
# Get predictions
predictions = model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(y_test, axis=1)

# Generate confusion matrix
conf_matrix = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=label_map.keys(), yticklabels=label_map.keys())
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

# Classification Report
print("Classification Report:")
print(classification_report(true_labels, predicted_labels, target_names=list(label_map.keys())))

# Plot Training History
plt.figure(figsize=(12, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import os
import zipfile
import numpy as np
import hashlib
from PIL import Image
from io import BytesIO

# Define paths
zip_file_path = "Final_ODIR_DataSet.zip"  # Path to ZIP file
new_zip_path = "Cleaned_ODIR_DataSet.zip"  # Path for the new cleaned dataset zip file

# Helper function to generate image hashes
def hash_image(img):
    img = img.convert("RGB")
    img = img.resize((224, 224))  # Resize for consistency
    img_array = np.array(img)
    img_hash = hashlib.md5(img_array.tobytes()).hexdigest()
    return img_hash

# Step 1: Remove duplicate images
def remove_duplicates(zip_file_path, img_size=(224, 224)):
    seen_hashes = set()
    unique_images = []
    unique_paths = []
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        for file_path in zip_ref.namelist():
            if file_path.endswith('.jpg') or file_path.endswith('.png'):
                # Calculate the hash for this image
                img_data = zip_ref.read(file_path)
                img = Image.open(BytesIO(img_data)).resize(img_size)
                img_hash = hash_image(img)
                
                # Add to the list if it's not a duplicate
                if img_hash not in seen_hashes:
                    seen_hashes.add(img_hash)
                    unique_images.append(img_data)
                    unique_paths.append(file_path)
                    
    return unique_images, unique_paths

# Step 2: Balance the Dataset (over-sample the smaller classes)
def balance_dataset(unique_images, unique_paths, img_size=(224, 224)):
    # Organize images by class
    class_dict = {}
    for path, img_data in zip(unique_paths, unique_images):
        class_name = path.split('/')[1]  # Assuming folder structure like /class_name/img.jpg
        if class_name not in class_dict:
            class_dict[class_name] = []
        class_dict[class_name].append(img_data)
    
    # Get maximum class size
    max_class_size = max(len(images) for images in class_dict.values())
    
    # Over-sample the smaller classes
    balanced_images = []
    balanced_paths = []
    for class_name, images in class_dict.items():
        while len(images) < max_class_size:
            images.extend(images)  # Duplicate images until balanced
        balanced_images.extend(images)
        balanced_paths.extend([f"{class_name}/{i}.jpg" for i in range(len(images))])
    
    return balanced_images, balanced_paths

# Step 3: Preprocess images (resize and normalize)
def preprocess_images_in_batches(images, batch_size=32):
    processed_images = []
    for i in range(0, len(images), batch_size):
        batch_images = images[i:i + batch_size]
        batch_processed_images = []
        for img_data in batch_images:
            img = Image.open(BytesIO(img_data))
            img = img.resize((224, 224))  # Resize for consistency
            img = np.array(img) / 255.0  # Normalize the image
            batch_processed_images.append(img)
        processed_images.append(batch_processed_images)
    return processed_images

# Step 4: Save the cleaned and preprocessed images to a new zip file
def save_to_zip(processed_images, paths, output_zip_path):
    # Create a new zip file
    with zipfile.ZipFile(output_zip_path, 'w') as zipf:
        for batch, batch_paths in zip(processed_images, paths):
            for img, path in zip(batch, batch_paths):
                # Save each image as a JPEG file in the new zip file
                img = Image.fromarray((img * 255).astype(np.uint8))
                with zipf.open(path, 'w') as imgf:
                    img.save(imgf, format="JPEG")

# Step 1: Remove duplicates
unique_images, unique_paths = remove_duplicates(zip_file_path)

# Step 2: Balance the dataset
balanced_images, balanced_paths = balance_dataset(unique_images, unique_paths)

# Step 3: Preprocess the images in batches (resize and normalize)
batch_size = 32  # Define a batch size
processed_images = preprocess_images_in_batches(balanced_images, batch_size=batch_size)

# Step 4: Save the cleaned dataset to a new zip file
save_to_zip(processed_images, balanced_paths, new_zip_path)

print(f"Cleaned dataset has been saved to {new_zip_path}")


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming data is loaded into variables x_train, y_train, x_val, y_val, x_test, y_test

# Compute class weights to handle imbalanced dataset
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(np.argmax(y_train, axis=1)),  # Get unique classes from y_train
    y=np.argmax(y_train, axis=1)  # The true labels for the training set
)

# Create a dictionary mapping class labels to their weights
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}
print("Class weights:", class_weight_dict)

# Load Pretrained VGG16 Model
vgg_base = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze Pretrained Layers
vgg_base.trainable = False

# Build the Model
model = models.Sequential([
    vgg_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.5),
    layers.Dense(len(class_weight_dict), activation="softmax")
])

# Compile the Model
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Train the Model for 100 epochs, using class weights to handle imbalance
history = model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=100,  # Increased number of epochs
    batch_size=32,
    class_weight=class_weight_dict  # Apply class weights here
)

# Evaluate on Test Data
test_loss, test_accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

# Save the model after training
model.save("Final_ODIR_Model_VGG16_100epochs_with_class_weights.h5")

# Confusion Matrix and Classification Report
# Get predictions on the test data
predictions = model.predict(x_test)
predicted_labels = np.argmax(predictions, axis=1)
true_labels = np.argmax(y_test, axis=1)

# Generate confusion matrix
conf_matrix = confusion_matrix(true_labels, predicted_labels)
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=list(class_weight_dict.keys()), yticklabels=list(class_weight_dict.keys()))
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.title("Confusion Matrix")
plt.show()

# Print the Classification Report
print("Classification Report:")
print(classification_report(true_labels, predicted_labels, target_names=list(class_weight_dict.keys())))

# Plot the training and validation accuracy and loss over epochs
plt.figure(figsize=(12, 5))

# Accuracy Plot
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

# Loss Plot
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()
